# EXO_M2_F05_CONTROL — Debug Alchemist Mode 2

```
╔══════════════════════════════════════════════════════════════════════════════╗
║    MODE 2 — M2_F05 ALCHEMIST — CONTROL PANEL — FUSION VISUELLE              ║
║                                                                              ║
║   Bible    : alchemist_schema.py — 7 piliers, 5 presets                     ║
║   Couleur  : match_color.py — Histogram Specification LAB (OpenCV)           ║
║   Grain    : grain_matcher.py — Bilateral decomposition + grain procédural   ║
║   Bloom    : bloom_engine.py — Luminance threshold → Gaussian → additive     ║
║   Netteté  : sharpness_transfer.py — Laplacian variance matching             ║
║                                                                              ║
║   Pipeline : match_color → grain → bloom → sharpness (CPU OpenCV)            ║
║   Loi R-01 : Copie étanche Mode 2 — zéro contamination Mode 1               ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

**Mission:** Tester unitairement les modules du pipeline Alchemist Mode 2

In [ ]:
#@title 🔗 [EXODUS] Drive + Session JSON
#@markdown Monte le Drive et lit exodus_session.json genere par EXO_LAUNCHER
from google.colab import drive
drive.mount('/content/drive')

import sys, json
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/EXODUS_V2"  #@param {type:"string"}
sys.path.insert(0, DRIVE_ROOT)

_session_path = Path(DRIVE_ROOT) / "exodus_session.json"
if _session_path.exists():
    with open(_session_path) as _f:
        EXODUS_SESSION = json.load(_f)
    print("OK exodus_session.json charge")
    print(f"  Mode     : {EXODUS_SESSION['mode']} --- {EXODUS_SESSION['mode_label']}")
    print(f"  Timestamp: {EXODUS_SESSION['timestamp']}")
    print(f"  Drive    : {EXODUS_SESSION['drive_root']}")
else:
    print("ATTENTION : exodus_session.json introuvable")
    print("   -> Lancer EXO_LAUNCHER.ipynb d'abord.")
    EXODUS_SESSION = {
        "mode": None, "mode_label": "UNKNOWN",
        "drive_root": DRIVE_ROOT, "status": "missing"
    }

## 1. Configuration

In [ ]:
import os
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path("/content/drive/MyDrive/EXODUS_V2")
# DRIVE_ROOT = Path("/home/EXODUS-V2")  # Local dev

FREGATE_ROOT = DRIVE_ROOT / "11_M2_F05_ALCHEMIST"
CODEBASE     = FREGATE_ROOT / "CODEBASE"
IN_RAW_FRAMES   = FREGATE_ROOT / "IN_RAW_FRAMES"
IN_SOURCE_REF   = FREGATE_ROOT / "IN_SOURCE_REF"
OUT_FINAL_FRAMES = FREGATE_ROOT / "OUT_FINAL_FRAMES"

sys.path.insert(0, str(CODEBASE))

print(f"Drive Root       : {DRIVE_ROOT}")
print(f"Frégate Root     : {FREGATE_ROOT}")
print(f"Codebase         : {CODEBASE}")
print(f"IN_RAW_FRAMES    : {IN_RAW_FRAMES}")
print(f"IN_SOURCE_REF    : {IN_SOURCE_REF}")
print(f"OUT_FINAL_FRAMES : {OUT_FINAL_FRAMES}")
print(f"Exists           : {FREGATE_ROOT.exists()}")

In [ ]:
!pip install -q numpy opencv-python-headless Pillow tqdm

## ⚡ GPU Check

In [ ]:
import subprocess

USE_CUDA = False
GPU_NAME = 'CPU'

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                              '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=10)
    if result.returncode == 0 and result.stdout.strip():
        GPU_NAME = result.stdout.strip().split(',')[0].strip()
        VRAM     = result.stdout.strip().split(',')[1].strip()
        USE_CUDA = True
        print(f'[GPU] {GPU_NAME} — {VRAM} — CUDA ACTIVE')
    else:
        print('[GPU] Aucun GPU détecté — mode CPU')
except Exception as e:
    print(f'[GPU] nvidia-smi non disponible ({e}) — mode CPU')

import cv2
cuda_cv2 = cv2.cuda.getCudaEnabledDeviceCount() if hasattr(cv2, 'cuda') else 0
print(f'[CV2] CUDA devices: {cuda_cv2}')

## 2. Test alchemist_schema

In [ ]:
from alchemist_schema import AlchemistSchema, PIPELINE_ORDER

schema = AlchemistSchema()
print('PIPELINE_ORDER:', PIPELINE_ORDER)

for preset in ['cinema_fusion', 'subtle_blend', 'neon_blast', 'raw_match', 'full_nuke']:
    valid, msg = schema.validate_pipeline_preset(preset)
    cfg = schema.get_pipeline_config(preset)
    print(f'  [{"OK" if valid else "ERR"}] {preset:15s} — {msg}')

## 3. Test bloom_engine

In [ ]:
import numpy as np
from bloom_engine import BloomEngine

engine = BloomEngine(verbose=True)
test_frame = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
result = engine.apply_bloom(test_frame, threshold=0.8, intensity=0.3, radius=51)
print(f'Input  : {test_frame.shape} {test_frame.dtype}')
print(f'Output : {result.shape} {result.dtype}')
print('bloom_engine OK')

## 4. Test sharpness_transfer

In [ ]:
from sharpness_transfer import SharpnessTransfer

st = SharpnessTransfer(verbose=True)
render = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
source = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
result = st.transfer(render, source, intensity=0.7)
print(f'sharpness_transfer OK — output: {result.shape} {result.dtype}')

## 5. Dry-run CLI

In [ ]:
PLAN_PATH = FREGATE_ROOT / "IN_PRODUCTION_PLAN" / "PRODUCTION_PLAN.JSON"

!python {CODEBASE}/EXO_M2_F05_ALCHEMIST.py \
  --production-plan {PLAN_PATH} \
  --dry-run --verbose